In [5]:
import polars as pl
from pathlib import Path

class DataPipeline:
    def __init__(self, input_path: str, output_path: str):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.invalid_values = ["none", "", "nan", "unknown"]
        
        # Define columns to drop
        self.drop_cols = [
            "about", "edu_list", "age", "first_major", "highest_degree",
            "duration_edu_certi", "first_edu_start", "end_edu_first",
            "experience", "duration_exp", "total_experience", "subtitle",
            "title_certi", "update_certi", "second_degree/certi_date",
            "update_duration", "first_certi", "last_certi",
            "last_major/certi_date", "total_time", "credential_url",
            "certi_standard", "certi_standard_stem", "first_industry_standard_1",
            "second_major/certi_standard", "second_major/certi_standard_stem",
            "first_major_standard_1", "last_certi_standard", "last_certi_standard_stem"
        ]

    def _is_valid_expr(self, col_name: str) -> pl.Expr:
        """Helper to check if a column contains meaningful data."""
        return (
            pl.col(col_name).is_not_null() & 
            (pl.col(col_name).str.strip_chars().str.to_lowercase().is_in(self.invalid_values).not_()) &
            (pl.col(col_name).str.strip_chars() != "")
        )

    def run(self):
        """Executes the lazy processing pipeline."""
        try:
            print(f"Initializing pipeline: {self.input_path}")
            
            # Use LazyFrame for memory efficiency
            lf = pl.scan_csv(self.input_path, infer_schema_length=0)

            # 1. Drop unnecessary columns
            lf = lf.drop(self.drop_cols)

            # 2. Define valid flags for each row
            is_temporal_valid = self._is_valid_expr("start_year") | self._is_valid_expr("start_date")
            is_industry_valid = self._is_valid_expr("first_industry") | self._is_valid_expr("current_industry")
            
            # 3. Create a mask: True if the row satisfies BOTH criteria
            lf = lf.with_columns(
                is_valid_row = (is_temporal_valid & is_industry_valid)
            )

            # 4. Window Function: If ANY row for this ID is valid, keep the whole ID
            # This prevents truncation of rows for a valid user.
            lf = lf.filter(
                pl.col("is_valid_row").any().over("id")
            ).drop("is_valid_row")

            print(f"Executing transformations and saving to {self.output_path}...")
            
            # Note: sink_csv does not support all operations (like window functions) 
            # in some Polars versions. If sink_csv fails, use collect().write_csv().
            lf.collect().write_csv(self.output_path)
            
            print("Pipeline execution successful.")

        except Exception as e:
            print(f"Critical Pipeline Error: {e}")

INPUT = "../Data/ha_original_data.csv"
OUTPUT = "../Data/ha_processed_data.csv"

pipeline = DataPipeline(INPUT, OUTPUT)
pipeline.run()

Initializing pipeline: ../Data/ha_original_data.csv
Executing transformations and saving to ../Data/ha_processed_data.csv...
Pipeline execution successful.


In [6]:
# unique individuals and metrics
import polars as pl

# 1. Initialize LazyFrame from the cleaned file
lazy_df = pl.scan_csv("../Data/ha_processed_data.csv")

# 2. Group by ID and count rows per person
# We use engine="streaming" to handle the high-cardinality grouping
counts_df = (
    lazy_df
    .group_by("id")
    .len()
    .collect(engine="streaming")
)

# 3. Calculate statistical metrics on the distribution
# 'len' represents the number of records associated with a single ID
stats = counts_df.select([
    pl.col("len").mean().alias("mean"),
    pl.col("len").median().alias("median"),
    pl.col("len").min().alias("min"),
    pl.col("len").max().alias("max"),
    pl.col("len").std().alias("std_dev"),
    pl.col("len").count().alias("unique_ids")
])

# 4. Output Results
res = stats.row(0, named=True)

print("--- RECORD DISTRIBUTION PER ID ---")
print(f"Total Unique Individuals: {res['unique_ids']:,}")
print("-" * 40)
print(f"Mean (Average):           {res['mean']:.2f} rows")
print(f"Median:                   {res['median']:.2f} rows")
print(f"Standard Deviation:       {res['std_dev']:.2f}")
print(f"Minimum per person:       {res['min']}")
print(f"Maximum per person:       {res['max']}")

--- RECORD DISTRIBUTION PER ID ---
Total Unique Individuals: 568,061
----------------------------------------
Mean (Average):           8.09 rows
Median:                   4.00 rows
Standard Deviation:       13.39
Minimum per person:       1
Maximum per person:       2261


In [7]:
import polars as pl

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

def analyze_completeness_polars(file_path):
    # 1. Use LazyFrame for memory efficiency
    lf = pl.scan_csv(file_path, infer_schema_length=10000)

    # 2. Heuristically identify JSON/Array columns to skip (using a sample)
    sample = lf.head(100).collect()
    json_cols = []
    for col in sample.columns:
        if col == "id":
            continue
        # Get first non-null value to check for JSON markers [ or {
        first_val = sample.select(pl.col(col).drop_nulls().first()).to_series()
        if not first_val.is_empty():
            val = first_val[0]
            if isinstance(val, str) and (val.startswith("[") or val.startswith("{")):
                json_cols.append(col)

    # 3. Define target columns (exclude id and JSON columns)
    target_cols = [c for c in sample.columns if c not in json_cols and c != "id"]

    # 4. Aggregate: Count unique IDs with at least one non-null/non-empty entry per column
    # Logic: Group by ID -> check if any row for that ID has valid data -> sum the 'Trues'
    analysis = (
        lf.group_by("id")
        .agg([
            (
                pl.col(c).is_not_null() & 
                (pl.col(c).cast(pl.Utf8).str.strip_chars() != "")
            ).any().alias(c)
            for c in target_cols
        ])
        .select([
            pl.col(c).sum().alias(c) for c in target_cols
        ])
        .collect()
    )

    # 5. Get total unique count for percentage calculation
    total_ids = lf.select(pl.col("id").n_unique()).collect().item()

    # 6. Format and display results
    results = []
    for col in target_cols:
        count = analysis[col][0]
        results.append({
            "Column": col,
            "Individuals_With_Data": count,
            "Percentage": (count / total_ids) * 100
        })

    summary_df = pl.DataFrame(results).sort("Individuals_With_Data", descending=True)
    
    print(f"Total Unique Individuals: {total_ids}")
    print(summary_df)

if __name__ == "__main__":
    analyze_completeness_polars("../Data/ha_processed_data.csv")

Total Unique Individuals: 568061
shape: (23, 3)
┌───────────────────┬───────────────────────┬────────────┐
│ Column            ┆ Individuals_With_Data ┆ Percentage │
│ ---               ┆ ---                   ┆ ---        │
│ str               ┆ i64                   ┆ f64        │
╞═══════════════════╪═══════════════════════╪════════════╡
│ industry          ┆ 568061                ┆ 100.0      │
│ industry standard ┆ 568061                ┆ 100.0      │
│ start_date        ┆ 568061                ┆ 100.0      │
│ end_date          ┆ 568061                ┆ 100.0      │
│ num_edu           ┆ 568061                ┆ 100.0      │
│ num_cer           ┆ 568061                ┆ 100.0      │
│ title             ┆ 568056                ┆ 99.99912   │
│ first_job_start   ┆ 568042                ┆ 99.996655  │
│ company           ┆ 567623                ┆ 99.922896  │
│ first_industry    ┆ 538491                ┆ 94.794573  │
│ degree            ┆ 512487                ┆ 90.216896  │
│ univer